<a href="https://colab.research.google.com/github/jpetr019/STAT108Project/blob/main/STAT_CS108_S26_Course_Project_Bias_Mitigation_4_(1)_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# STAT/CS108 S26 Course Project

### Team Details

- Teammate 1: Name
- Teammate 2: Name
- Teammate 3: Name

---

# Installs

In [37]:
# [INSERT CODE HERE to install necessary packages]
!pip install ucimlrepo
!pip install fairlearn

# Imports

In [38]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
import collections
from pprint import pprint
from sklearn.model_selection import train_test_split
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.metrics import accuracy_score

## Add additional imports needed for your project here.

from ucimlrepo import fetch_ucirepo

# Loading dataset
_(same as previous milestone, copy-paste)_

In [39]:
# Load your selected dataset

# fetch dataset
adult = fetch_ucirepo(id=2)

# data (as pandas dataframes)
X = adult.data.features
y = adult.data.targets

sensitive_feature_colname = ['sex', 'race'] # sensitive feature name

# Make sensitive features-based group labels
group_labels = (X['sex'].astype(str) + '_' + X['race'].astype(str))

# Print some stats
print(f"No. of samples: {X.shape[0]}")
print(f"No. of features: {X.shape[1]}")
print(f"Group Counts: {dict(collections.Counter(group_labels))}")

No. of samples: 48842
No. of features: 14
Group Counts: {'Male_White': 28735, 'Male_Black': 2377, 'Female_Black': 2308, 'Female_White': 13027, 'Male_Asian-Pac-Islander': 1002, 'Male_Amer-Indian-Eskimo': 285, 'Female_Other': 155, 'Female_Asian-Pac-Islander': 517, 'Female_Amer-Indian-Eskimo': 185, 'Male_Other': 251}


# Preparing dataset
_(same as previous milestone, copy-paste)_

In [40]:
# Some subset of following dataset preparation steps may be necessary depending on your dataset,
# 1. Drop unnecessary features
# 2. Handle missing data
# 3. Encode categorical features
# 4. Normalize numerical features
# 5. Encode target (if your task is classification)

# [INSERT YOUR CODE HERE]
# creating copies

X = X.copy()
y = y.squeeze().copy()

# replacing missing value markers with NaN
X = X.replace('?', np.nan)

# dropping rows with missing values
mask = X.notna().all(axis=1) & y.notna()
X = X.loc[mask]
y = y.loc[mask]
group_labels = group_labels.loc[mask]

# encoding target labels (salary) as 0/1

y = y.astype(str).str.strip().str.replace('.', '', regex=False)
y = y.map({'<=50K': 0, '>50K': 1})

# one hot encoding for categorical features since logisitc regression requires numerical values
X = pd.get_dummies(X, drop_first=True) # adding drop_first to avoid redundant columns with sex_Female: 0 and sex_Male: 1 that communicate the same thing

# Note: X and y have been modified before the following lines of code!
print(f"No. of samples AFTER cleaning: {X.shape[0]}")
assert X.shape[0] == y.shape[0] == group_labels.shape[0] ## Ensure that the target and group_labels have been updated if some samples were removed during cleaning.
print(f"No. of features AFTER encoding: {X.shape[1]}")

No. of samples AFTER cleaning: 45222
No. of features AFTER encoding: 96


# Getting training and testing sets

Note: Train-test split is made **ONCE** to obtain the _training set_ and the _testing set_ and every teammate will use the training set to train their baseline model and test the trained model using the testing set. **NEVER** modify the testing set once it has been created.
Therefore, the following code cell does not need to be edited.

_(same as previous milestone, copy-paste)_

In [41]:
X_train, X_test, \
  y_train, y_test, \
    group_labels_train, group_labels_test = train_test_split(X, y, group_labels,
                                                             test_size=0.2, random_state=42)

print(f"No. of training samples: {X_train.shape[0]}")
print(f"No. of testing samples: {X_test.shape[0]}")

# Delete X, y and group_label variables to make sure they are not used later on.
del X
del y
del group_labels

No. of training samples: 36177
No. of testing samples: 9045


# Setting up evaluation metrics
Note: The same evaluation function will be used by all teammates.

_(same as previous milestone, copy-paste)_

In [42]:
from sklearn.metrics import accuracy_score, f1_score
from fairlearn.metrics import (
    MetricFrame,
    selection_rate,
    true_positive_rate,
    false_positive_rate,
    demographic_parity_difference,
    demographic_parity_ratio
)

def evaluate_model(y_test, y_pred, g_labels):
  """
  Evaluate the performance of your trained model on the testing set.

  Parameters
  ----------
  y_test : array-like
    The true targets of the testing set.
  y_pred : array-like
    The predicted targets of the testing set.
  g_labels : array-like
    The group labels of the testing set.

  Returns
  -------
  results : dict
    A dictionary containing the evaluation results.

    Example:
      For classification task, the task-specific performance metrics like {'accuracy': <value>, 'f1_score': <value>, ...}
      and fairness metrics like {'demographic_parity': <value>, 'equalized_odds': <value>, ...}.

  """
  results = {}

  # Note: These metrics will be calculated for - 1. the full testing set, 2. individual groups.
  # Task-specific performance metrics
  # [INSERT CODE HERE for performance metrics appropriate for your task]

  # ----------------------------
  # Overall performance metrics
  # ----------------------------
  results["overall"] = {
      "accuracy": accuracy_score(y_test, y_pred),
      "f1_score": f1_score(y_test, y_pred, zero_division=0)
  }

  # ----------------------------
  # Group-level metrics
  # ----------------------------
  metric_frame = MetricFrame(
      metrics={
          "accuracy": accuracy_score,
          "f1_score": lambda y_true, y_pred: f1_score(y_true, y_pred, zero_division=0),
          "selection_rate": selection_rate,
          "true_positive_rate": true_positive_rate,
          "false_positive_rate": false_positive_rate
      },
      y_true=y_test,
      y_pred=y_pred,
      sensitive_features=g_labels
  )

  results["by_group"] = metric_frame.by_group

  # Fairness metric
  # [INSERT CODE HERE for fairness metric appropriate for your task]

  # ----------------------------
  # Fairness metrics
  # ----------------------------

  # Demographic parity:
  # difference in positive prediction rates across groups
  results["demographic_parity_difference"] = demographic_parity_difference(
      y_test,
      y_pred,
      sensitive_features=g_labels
  )

  # Disparate impact ratio:
  # ratio of lowest selection rate to highest selection rate
  results["disparate_impact_ratio"] = demographic_parity_ratio(
      y_test,
      y_pred,
      sensitive_features=g_labels
  )

  # Equal opportunity:
  # difference in true positive rates across groups
  results["equal_opportunity_difference"] = metric_frame.difference()["true_positive_rate"]

  # Equalized odds:
  # max of TPR difference and FPR difference
  tpr_diff = metric_frame.difference()["true_positive_rate"]
  fpr_diff = metric_frame.difference()["false_positive_rate"]

  results["equalized_odds_difference"] = max(tpr_diff, fpr_diff)


  return results

# Training baseline models (INDIVIDUAL CONTRIBUTION)

In [43]:
## A place to save all teammates's baseline results
all_baseline_results = [] ## DO NOT EDIT

## Teammate 1

In [44]:
# Select a model and train it on the training set
# [INSERT YOUR CODE HERE]
model = LogisticRegression(max_iter=1000)
# Make predictions on the testing set and store them in y_pred
model.fit(X_train, y_train)
y_pred = model.predict(X_test) # [INSERT CODE HERE]

# Evaluate testing set predictions using evaluate_model()
results = evaluate_model(y_test, y_pred, group_labels_test)

# Save your results to all_baseline_results
results['teammate'] = 'Teammate 1'
results['experiment_type'] = 'baseline'
results['predictor_model'] = 'Logistic Regression' #[INSERT MODEL NAME HERE]
results['mitigation_strategy'] = 'NONE' ## DO NOT EDIT: This is pre-mitigation baseline
all_baseline_results.append(results)

print(results)

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


{'overall': {'accuracy': 0.8475400773908237, 'f1_score': 0.6693838408055622}, 'by_group':                            accuracy  f1_score  selection_rate  \
sensitive_feature_0                                             
Female_Amer-Indian-Eskimo  0.925000  0.800000        0.175000   
Female_Asian-Pac-Islander  0.880000  0.470588        0.133333   
Female_Black               0.957684  0.595745        0.042316   
Female_Other               1.000000  1.000000        0.090909   
Female_White               0.913062  0.595745        0.088186   
Male_Amer-Indian-Eskimo    0.862069  0.600000        0.172414   
Male_Asian-Pac-Islander    0.744565  0.675862        0.434783   
Male_Black                 0.857143  0.544118        0.122120   
Male_Other                 0.833333  0.588235        0.166667   
Male_White                 0.809818  0.687980        0.275623   

                           true_positive_rate  false_positive_rate  
sensitive_feature_0                                         

## Teammate 2

In [45]:
# Select a model and train it on the training set
# [INSERT YOUR CODE HERE]

# Make predictions on the testing set and store them in y_pred
y_pred = ... # [INSERT CODE HERE]

# Evaluate testing set predictions using evaluate_model()
results = evaluate_model(y_test, y_pred, group_labels_test)

# Save your results to all_baseline_results
results['teammate'] = 'Teammate 2'
results['experiment_type'] = 'baseline'
results['predictor_model'] = ... #[INSERT MODEL NAME HERE]
results['mitigation_strategy'] = 'NONE' ## DO NOT EDIT: This is pre-mitigation baseline
all_baseline_results.append(results)

pprint(results)

InvalidParameterError: The 'y_pred' parameter of accuracy_score must be an array-like or a sparse matrix. Got Ellipsis instead.

## Teammate 3

In [ ]:
# Select a model and train it on the training set
# [INSERT YOUR CODE HERE]

# Make predictions on the testing set and store them in y_pred
y_pred = ... # [INSERT CODE HERE]

# Evaluate testing set predictions using evaluate_model()
results = evaluate_model(y_test, y_pred, group_labels_test)

# Save your results to all_baseline_results
results['teammate'] = 'Teammate 3'
results['experiment_type'] = 'baseline'
results['predictor_model'] = ... #[INSERT MODEL NAME HERE]
results['mitigation_strategy'] = 'NONE' ## DO NOT EDIT: This is pre-mitigation baseline
all_baseline_results.append(results)

pprint(results)

# Mitigating Bias (INDIVIDUAL CONTRIBUTION)



In [ ]:
## A place to save all teammates' post-mitigation results
all_mitigated_results = [] ## DO NOT EDIT

## Teammate 1

In [ ]:
# Implement your bias mitigation strategy
## If you chose preprocessing, you will train a new version of your predictor model with new/modified inputs.
## If you chose inprocessing, you will train a new version of your predictor with modified learning objective (loss function).
## If you chose postprocessing, you will implement strategies to modify the predictions (y_pred) of the trained baseline predictor model from the previous milestone without training any new version of the predictor model.

# [INSERT CODE HERE]

# Make predictions on the testing set and store them in y_pred_mitigate
y_pred_mitigated = ... # [INSERT CODE HERE]

# Evaluate testing set predictions using evaluate_model()
results_mitigated = evaluate_model(y_test, y_pred_mitigated, group_labels_test)

# Save your results to all_mitigated_results
results_mitigated['teammate'] = 'Teammate 1'
results_mitigated['experiment_type'] = 'post-mitigation'
results_mitigated['predictor_model'] = ... #[INSERT MODEL NAME HERE]
results_mitigated['mitigation_strategy'] = ... #[INSERT STRATEGY TYPE HERE: 'preprocessing', 'inprocessing', 'postprocessing']
all_mitigated_results.append(results_mitigated)

pprint(results_mitigated)

### Teammate 1's Conclusions
[Briefly describe findings and conclusions here. Compare post-mitigation results with baseline results for your model. What is the % improvement in performance post-mitigation?  ]

## Teammate 2

In [ ]:
# Implement your bias mitigation strategy
## If you chose preprocessing, you will train a new version of your predictor model with new/modified inputs.
## If you chose inprocessing, you will train a new version of your predictor with modified learning objective (loss function).
## If you chose postprocessing, you will implement strategies to modify the predictions (y_pred) of the trained baseline predictor model from the previous milestone without training any new version of the predictor model.

# [INSERT CODE HERE]

# Make predictions on the testing set and store them in y_pred_mitigate
y_pred_mitigated = ... # [INSERT CODE HERE]

# Evaluate testing set predictions using evaluate_model()
results_mitigated = evaluate_model(y_test, y_pred_mitigated, group_labels_test)

# Save your results to all_mitigated_results
results_mitigated['teammate'] = 'Teammate 2'
results_mitigated['experiment_type'] = 'post-mitigation'
results_mitigated['predictor_model'] = ... #[INSERT MODEL NAME HERE]
results_mitigated['mitigation_strategy'] = ... #[INSERT STRATEGY TYPE HERE: 'preprocessing', 'inprocessing', 'postprocessing']
all_mitigated_results.append(results_mitigated)

print(results_mitigated)

### Teammate 2's Conclusions
[Briefly describe findings and conclusions here. Compare post-mitigation results with baseline results for your model. What is the % improvement in performance post-mitigation?]


## Teammate 3

In [ ]:
# Implement your bias mitigation strategy
## If you chose preprocessing, you will train a new version of your predictor model with new/modified inputs.
## If you chose inprocessing, you will train a new version of your predictor with modified learning objective (loss function).
## If you chose postprocessing, you will implement strategies to modify the predictions (y_pred) of the trained baseline predictor model from the previous milestone without training any new version of the predictor model.

# [INSERT CODE HERE]

# Make predictions on the testing set and store them in y_pred_mitigate
y_pred_mitigated = ... # [INSERT CODE HERE]

# Evaluate testing set predictions using evaluate_model()
results_mitigated = evaluate_model(y_test, y_pred_mitigated, group_labels_test)

# Save your results to all_mitigated_results
results_mitigated['teammate'] = 'Teammate 3'
results_mitigated['experiment_type'] = 'post-mitigation'
results_mitigated['predictor_model'] = ... #[INSERT MODEL NAME HERE]
results_mitigated['mitigation_strategy'] = ... #[INSERT STRATEGY TYPE HERE: 'preprocessing', 'inprocessing', 'postprocessing']
all_mitigated_results.append(results_mitigated)

print(results_mitigated)

### Teammate 3's Conclusions
[Briefly describe findings and conclusions here. Compare post-mitigation results with baseline results for your model. What is the % improvement in performance post-mitigation?]


# Conclusions
_(new in this milestone)_


In [ ]:
# Collect all the results in one table.
overall_results = pd.concat([pd.DataFrame(all_baseline_results), pd.DataFrame(all_mitigated_results)])
overall_results ## Note: The table displayed below in this starter notebook is for your reference, your team's table will be slightly different (e.g. different metrics, no.of sensitive attribute-based groups, actual values, etc.) upon successful completion of this notebook.

[Briefly describe overall findings and conclusions here. Which mitigation strategy resulted in most improvement? Which resulted in the least improvement? Visualize the results with some informative plots. (Hint: Use the `overall_results` table).]

# References

[List the references you used to complete this milestone here.]
- Teammate 1: The UCI Machine Learning Repository Adult dataset page was used to import the dataset directly in Python using the ucimlrepo package and fetch_ucirepo(id=2). Becker, B., & Kohavi, R. (1996). Adult [Dataset]. UCI Machine Learning Repository. https://doi.org/10.24432/C5XW20
Evaluation metric code was adapted from prior course discussion notebooks, especially Week 4’s Fairlearn metric examples. The code was modified for the Adult dataset and our selected sensitive feature groups.
- Teammate 2:
- Teammate 3:

# Disclosures

[Disclose use of generative AI and similar tools here.]
- Teammate 1: ChatGPT was used for clarification of assignment concepts and wording feedback, but not to generate final code or complete the assignment.
- Teammate 2:
- Teammate 3: